## Assignment 6

### Perform bag-of-words approach (count occurrence, normalized count occurrence), TF-IDF on data. Create embeddings using Word2Vec

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
docs = [
    "machine learning is fun",
    "deep learning is powerful",
    "machine learning and deep learning"
]

docs

['machine learning is fun',
 'deep learning is powerful',
 'machine learning and deep learning']

In [3]:
cv = CountVectorizer()
bow = cv.fit_transform(docs)

bow_df = pd.DataFrame(
    bow.toarray(),
    columns=cv.get_feature_names_out()
)

bow_df

,and,deep,fun,is,learning,machine,powerful
0,0,0,1,1,1,1,0
1,0,1,0,1,1,0,1
2,1,1,0,0,2,1,0


In [4]:
tfidf = TfidfVectorizer()
tfidf_mat = tfidf.fit_transform(docs)

tfidf_df = pd.DataFrame(
    tfidf_mat.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df

,and,deep,fun,is,learning,machine,powerful
0,0.000000,0.000000,0.631745,0.480458,0.373119,0.480458,0.000000
1,0.000000,0.480458,0.000000,0.480458,0.373119,0.000000,0.631745
2,0.530587,0.403525,0.000000,0.000000,0.626747,0.403525,0.000000


In [5]:
tokens = [d.split() for d in docs]

vocab = sorted(set(word for sent in tokens for word in sent))

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

vocab

['and', 'deep', 'fun', 'is', 'learning', 'machine', 'powerful']

In [6]:
window = 2
pairs = []

for sent in tokens:
    for i,word in enumerate(sent):
        for j in range(max(0,i-window), min(len(sent), i+window+1)):
            if i != j:
                pairs.append((word, sent[j]))

pairs[:10]

[('machine', 'learning'),
 ('machine', 'is'),
 ('learning', 'machine'),
 ('learning', 'is'),
 ('learning', 'fun'),
 ('is', 'machine'),
 ('is', 'learning'),
 ('is', 'fun'),
 ('fun', 'learning'),
 ('fun', 'is')]

In [7]:
V = len(vocab)
N = 5

def one_hot(i):
    vec = np.zeros(V)
    vec[i] = 1
    return vec

np.random.seed(0)
W = np.random.randn(V,N)
W2 = np.random.randn(N,V)

lr = 0.05

for epoch in range(300):
    for w,c in pairs:
        x = one_hot(word2idx[w])
        h = W.T @ x
        u = W2.T @ h
        y_pred = np.exp(u)/np.sum(np.exp(u))

        y = one_hot(word2idx[c])
        e = y_pred - y

        dW2 = np.outer(h,e)
        dW = np.outer(x, W2 @ e)

        W2 -= lr*dW2
        W -= lr*dW

In [8]:
embeddings = pd.DataFrame(W, index=vocab)
embeddings

,0,1,2,3,4
and,1.274547,0.556458,0.207970,1.587744,1.302290
deep,-0.779812,-0.836807,-1.369881,0.812719,0.839303
fun,-0.603850,0.804795,-1.826328,0.093988,0.931419
is,1.493059,-0.074090,-0.025022,-0.169619,0.483616
learning,-0.874754,0.490739,0.862441,0.024527,-0.250248
machine,-0.713651,-0.706284,-1.316438,0.759653,0.320023
powerful,-0.357874,0.497849,-2.338112,-0.798986,0.503989
